In [1]:
import numpy as np

# Generate mixture data


In [53]:
"""
Mallows model with tied (block) consensus – samples strict linear orders.

Model
-----
    P(σ | ρ₀, α) ∝ exp(−α · d_K(σ, ρ₀))

where σ is a strict ranking, ρ₀ is a consensus with ties (given as an
ordered list of blocks), α ≥ 0 is the dispersion parameter, and d_K is
the Kendall-tau distance generalized for ties:

  * A pair (i,j) where ρ₀ strictly orders i above j contributes 1 if σ
    reverses them, 0 otherwise.
  * A pair (i,j) tied in ρ₀ contributes 0 regardless of σ.

Sampling Strategy
-----------------
Adjacent-transposition Gibbs sampler with O(1) incremental distance updates.
At each step we pick a random adjacent pair (σ[i], σ[i+1]) and decide whether
to swap using only the delta in Kendall distance (not recomputing full distance):

  * O(1) check: if pair is from same block (tied), swap with prob 0.5
  * O(1) check: if swapping fixes disagreement (Δ = −1), always swap
  * O(1) check: if swapping creates disagreement (Δ = +1), swap with prob exp(−α)

This satisfies detailed balance & is ergodic. Key efficiency: avoiding O(n²)
full distance recomputation at each step.
"""

from __future__ import annotations

import numpy as np
from typing import Sequence
from collections import defaultdict


# ---------------------------------------------------------------------------
# Helper: Build block mapping (pre-compute once, reuse)
# ---------------------------------------------------------------------------

def _build_block_mapping(blocks: list[list[int]]) -> dict[int, int]:
    """Efficiently build item-to-block mapping. O(n) cost, reused throughout."""
    block_of = {}
    for b_idx, block in enumerate(blocks):
        for item in block:
            block_of[item] = b_idx
    return block_of


# ---------------------------------------------------------------------------
# Kendall distance (diagnostic only, O(n²))
# ---------------------------------------------------------------------------

def kendall_distance_tied(
    sigma: Sequence[int],
    block_of: dict[int, int],
) -> int:
    """
    Kendall-tau distance between strict ranking σ and tied consensus ρ₀.
    
    Note: O(n²) computation. Used for diagnostics/validation only.
    During sampling, we use incremental O(1) updates via _swap_delta instead.
    
    Parameters
    ----------
    sigma : sequence of int
        Strict ranking (permutation).
    block_of : dict[int, int]
        Item-to-block mapping (typically pre-computed once per cluster).
    
    Returns
    -------
    int
        Kendall-tau distance.
    """
    pos_in_sigma = {item: pos for pos, item in enumerate(sigma)}

    dist = 0
    items = list(block_of.keys())
    n = len(items)
    for a in range(n):
        for b in range(a + 1, n):
            i, j = items[a], items[b]
            # Tied pairs contribute 0 to distance
            if block_of[i] == block_of[j]:
                continue
            # Check if discordant
            if block_of[i] < block_of[j]:
                if pos_in_sigma[i] > pos_in_sigma[j]:
                    dist += 1
            else:
                if pos_in_sigma[j] > pos_in_sigma[i]:
                    dist += 1
    return dist


# ---------------------------------------------------------------------------
# Gibbs sampler: O(1) per step via incremental distance
# ---------------------------------------------------------------------------

def _swap_delta(
    sigma: list[int], pos: int, block_of: dict[int, int]
) -> int:
    """
    Change in Kendall distance from swapping sigma[pos] ↔ sigma[pos+1].
    
    O(1) operation: only check the single adjacent pair, not full distance.
    
    Returns
    -------
    int
        0: tied pair (no distance contribution)
        -1: swap improves (fixes disagreement)
        +1: swap worsens (creates disagreement)
    """
    i, j = sigma[pos], sigma[pos + 1]
    bi, bj = block_of[i], block_of[j]
    if bi == bj:
        # Tied pair—contributes 0 regardless of order
        return 0
    # Discordant pair: check ordering
    return 1 if bi < bj else -1


def _gibbs_step(
    sigma: list[int],
    block_of: dict[int, int],
    alpha: float,
    rng: np.random.Generator,
) -> None:
    """
    One Gibbs step: pick random adjacent pair, decide whether to swap (in-place).
    
    O(1) per step using incremental _swap_delta computation.
    """
    n = len(sigma)
    if n < 2:
        return
    pos = int(rng.integers(0, n - 1))
    delta = _swap_delta(sigma, pos, block_of)
    if delta <= 0:
        # Tied pair or improves: always swap
        sigma[pos], sigma[pos + 1] = sigma[pos + 1], sigma[pos]
    else:
        # Worsens: swap with prob exp(-α)
        if rng.random() < np.exp(-alpha):
            sigma[pos], sigma[pos + 1] = sigma[pos + 1], sigma[pos]


def sample_mallows_tied(
    blocks: list[list[int]],
    alpha: float,
    rng: np.random.Generator,
    *,
    n_samples: int = 1,
    burn_in: int | None = None,
    thin: int | None = None,
) -> list[int] | list[list[int]]:
    """
    Sample strict ranking(s) from Mallows with tied consensus.
    
    Uses adjacent-transposition Gibbs sampling where each step is O(1)
    via incremental distance computation (avoiding O(n²) recomputation).

    Parameters
    ----------
    blocks : list[list[int]]
        Consensus ρ₀ as ordered blocks (best-first).
    alpha : float
        Dispersion parameter (≥ 0).
    rng : numpy.random.Generator
    n_samples : int
        How many rankings to draw.
    burn_in : int or None
        MCMC burn-in steps.  Default: 10·n².
    thin : int or None
        Steps between samples.  Default: 5·n².

    Returns
    -------
    list[int] if n_samples == 1, else list[list[int]]
        Sampled ranking(s).
    """
    if alpha < 0:
        raise ValueError("alpha must be >= 0")

    # Pre-compute block mapping once (reused throughout all steps)
    block_of = _build_block_mapping(blocks)

    n = sum(len(b) for b in blocks)
    if n == 0:
        return [] if n_samples == 1 else [[] for _ in range(n_samples)]

    n2 = n * n
    if burn_in is None:
        burn_in = 10 * n2
    if thin is None:
        thin = 5 * n2

    # Initialize: sort by block, shuffle within each block
    sigma: list[int] = []
    for block in blocks:
        shuffled = list(block)
        rng.shuffle(shuffled)
        sigma.extend(shuffled)

    # Burn-in phase: discard these samples
    for _ in range(burn_in):
        _gibbs_step(sigma, block_of, alpha, rng)

    # Collect samples
    samples = []
    for s in range(n_samples):
        if s > 0:
            # Thin: wait 'thin' steps between samples for independence
            for _ in range(thin):
                _gibbs_step(sigma, block_of, alpha, rng)
        samples.append(list(sigma))

    return samples[0] if n_samples == 1 else samples

In [54]:


# ---------------------------------------------------------------------------
# Batch sampler: efficiently draw many samples from single chain
# ---------------------------------------------------------------------------

def sample_mallows_tied_batch(
    blocks: list[list[int]],
    alpha: float,
    rng: np.random.Generator,
    n_samples: int,
    burn_in: int | None = None,
    thin: int | None = None,
) -> list[list[int]]:
    """
    Efficiently draw n_samples rankings from a single Markov chain.
    
    More efficient than calling sample_mallows_tied(n_samples=1) repeatedly,
    as it reuses the chain state and performs burn-in only once.

    Parameters
    ----------
    blocks : list[list[int]]
        Consensus blocks.
    alpha : float
        Dispersion parameter.
    rng : numpy.random.Generator
    n_samples : int
        Number of samples to draw.
    burn_in, thin : int or None
        MCMC tuning parameters.

    Returns
    -------
    list[list[int]]
        Always returns list of lists (even if n_samples=1), for consistency.
    """
    result = sample_mallows_tied(
        blocks, alpha, rng,
        n_samples=n_samples, burn_in=burn_in, thin=thin,
    )
    if n_samples == 1:
        return [result]
    return result


# ---------------------------------------------------------------------------
# Full data generator: mixture of Mallows with tied consensus
# ---------------------------------------------------------------------------

def generate_mixture_data(
    n_assessors: int = 100,
    n_items: int = 10,
    C: int = 3,
    seed: int = 1,
    alpha: float = 3.0,
    tau: np.ndarray | None = None,
    n_blocks: int | None = None,
    n_blocks_range: tuple[int, int] = (3, 6),
    min_block_size: int = 1,
    burn_in: int | None = None,
    thin: int | None = None,
) -> tuple[list[list[list[int]]], list[float], list[int], list[list[int]]]:
    """
    Generate a mixture of Mallows models with tied consensus rankings.
    
    Efficiency: Uses batch sampling grouped by cluster to share MCMC chains
    within each cluster, avoiding redundant warm-up.

    Each cluster c has a tied consensus ρ₀ᶜ and shared dispersion α.
    Each assessor is assigned to a cluster via τ, then their strict
    ranking is sampled from Mallows(ρ₀ᶜ, α).

    Parameters
    ----------
    n_assessors : int
        Number of assessors.
    n_items : int
        Number of items.
    C : int
        Number of clusters.
    seed : int
        Random seed.
    alpha : float
        Mallows dispersion.  Higher → closer to consensus.
    tau : array-like or None
        Mixture weights (length C).  None → uniform.
    n_blocks : int or None
        Fixed number of blocks per cluster. If None, sampled randomly.
    n_blocks_range : tuple[int, int]
        (min, max) number of blocks to sample per cluster (if n_blocks=None).
    min_block_size : int
        Minimum size of each block.
    burn_in, thin : int or None
        MCMC parameters.  None → automatic (10n² and 5n²).

    Returns
    -------
    tuple
        (true_blocks, tau, z_true, rankings)
        - true_blocks[c] = list of blocks defining consensus for cluster c
        - tau = mixture weights (normalized)
        - z_true = cluster assignment (one per assessor)
        - rankings = sampled strict rankings (permutation per assessor)
    """
    rng = np.random.default_rng(seed)

    if n_items < 1:
        raise ValueError("n_items must be >= 1")
    if C < 1:
        raise ValueError("C must be >= 1")
    if min_block_size < 1:
        raise ValueError("min_block_size must be >= 1")

    # ---- Normalize mixture weights ----
    if tau is None:
        tau_arr = np.ones(C) / C
    else:
        tau_arr = np.asarray(tau, dtype=float)
        if tau_arr.shape != (C,):
            raise ValueError(f"tau must have length C={C}")
        if tau_arr.sum() <= 0:
            raise ValueError("tau must sum to a positive value")
        tau_arr = tau_arr / tau_arr.sum()

    # ---- Helper functions for block generation ----
    def choose_k() -> int:
        """Choose number of blocks for this cluster."""
        if n_blocks is not None:
            k = int(n_blocks)
        else:
            lo, hi = n_blocks_range
            if lo < 1 or hi < lo:
                raise ValueError("n_blocks_range must be (low>=1, high>=low)")
            k = int(rng.integers(lo, hi + 1))
        # Cap k so each block can have at least min_block_size items
        k_max = n_items // min_block_size
        if k_max < 1:
            return 1
        return max(1, min(k, k_max))

    def sizes_no_empty(total: int, k: int, min_part: int) -> list[int]:
        """Allocate 'total' items into k blocks, each with >= min_part items."""
        base = np.full(k, min_part, dtype=int)
        remaining = total - k * min_part
        if remaining < 0:
            raise ValueError(
                f"Impossible: total={total}, k={k}, min_part={min_part}"
            )
        # Randomly distribute remaining items
        extras = rng.multinomial(remaining, np.ones(k) / k)
        sizes = base + extras
        rng.shuffle(sizes)
        return sizes.tolist()

    def make_cluster_blocks() -> list[list[int]]:
        """Generate one random block partition for a cluster."""
        k = choose_k()
        sizes = sizes_no_empty(n_items, k, min_block_size)
        items = rng.permutation(n_items).tolist()
        blocks_list: list[list[int]] = []
        idx = 0
        for sz in sizes:
            block = items[idx : idx + sz]
            assert len(block) > 0, "Empty block should be impossible"
            blocks_list.append(block)
            idx += sz
        assert idx == n_items, "Did not consume all items"
        assert len({x for b in blocks_list for x in b}) == n_items, \
            "Items repeated or missing"
        return blocks_list

    # ---- Generate block structures and cluster assignments ----
    true_blocks = [make_cluster_blocks() for _ in range(C)]
    z_true = rng.choice(C, size=n_assessors, p=tau_arr).tolist()

    # ---- Efficient batch sampling: group assessors by cluster ----
    # Within each cluster, sample from a single chain to avoid redundant burn-in
    cluster_indices: dict[int, list[int]] = defaultdict(list)
    for i, z in enumerate(z_true):
        cluster_indices[z].append(i)

    rankings: list[list[int] | None] = [None] * n_assessors
    for c, indices in cluster_indices.items():
        # Sample len(indices) rankings from one chain for cluster c
        sampled = sample_mallows_tied_batch(
            true_blocks[c], alpha, rng,
            n_samples=len(indices),
            burn_in=burn_in, thin=thin,
        )
        for idx, s in zip(indices, sampled):
            rankings[idx] = s

    return true_blocks, tau_arr.tolist(), z_true, rankings  # type: ignore


# Estimate p

In [5]:
"""
Estimation of the penalty parameter p for the K(p) distance
in the Weak-Mallows Model.

Two approaches:
1. Simple confidence-interval estimate (Section 4.1, eq. near bottom of p.6)
2. Data-informed Bayesian estimate using posterior tie probabilities,
   Bradley-Terry model for alpha, Pitman-Yor prior for pi_1,
   and misclassification loss minimisation (Section 4.1, eqs. 19-26).

Extended to multiple clusters with cluster-weighted averaging.

Author: Implementation based on Jonas Nordstrøm (Dec 2025) working notes.
"""

from __future__ import annotations

import numpy as np
from scipy.special import comb, gammaln
from scipy.optimize import minimize_scalar, minimize
from typing import Optional
import warnings


# ---------------------------------------------------------------------------
# 1. Simple confidence-interval estimate
# ---------------------------------------------------------------------------

def estimate_p_simple(N: int) -> float:
    """
    Quick estimate of p from the 95% CI of the binomial proportion
    under the null hypothesis that a pair is tied (r_ij ~ Binom(N, 0.5)).

    p = (sqrt(N) - 1.96) / (2 * sqrt(N))

    Parameters
    ----------
    N : int
        Number of assessors (observations).

    Returns
    -------
    float
        Estimated p in (0, 0.5). Clipped to (1e-6, 0.5 - 1e-6).
    """
    sqN = np.sqrt(N)
    p = (sqN - 1.96) / (2.0 * sqN)
    return float(np.clip(p, 1e-6, 0.5 - 1e-6))


def estimate_p_simple_multiclusters(
    N_total: int,
    C: int,
) -> float:
    """
    Simple CI estimate corrected for clustering: use the average cluster
    size N_avg = N_total / C in place of N.

    Parameters
    ----------
    N_total : int
        Total number of assessors.
    C : int
        Number of clusters.

    Returns
    -------
    float
        Estimated p.
    """
    N_avg = max(N_total / C, 2)
    return estimate_p_simple(int(round(N_avg)))


# ---------------------------------------------------------------------------
# 2. Data-informed Bayesian estimation
# ---------------------------------------------------------------------------

# 2a. Pairwise summary statistics -----------------------------------------

def pairwise_counts(rankings: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    From an (N x n) matrix of strict rankings, compute pairwise counts.

    Parameters
    ----------
    rankings : ndarray of shape (N, n)
        rankings[l, i] = rank of item i by assessor l (lower = better).
        Must be strict (no ties in observed rankings).

    Returns
    -------
    s : ndarray (n, n)   s[i,j] = # times item i ranked above j  (r(i) < r(j))
    t : ndarray (n, n)   t[i,j] = # ties (always 0 for strict rankings)
    N : int              number of assessors
    """
    N, n = rankings.shape
    s = np.zeros((n, n), dtype=int)
    for l in range(N):
        r = rankings[l]
        # i ranked above j means r[i] < r[j]
        for i in range(n):
            for j in range(n):
                if i != j and r[i] < r[j]:
                    s[i, j] += 1
    t = np.zeros((n, n), dtype=int)  # strict rankings => no ties
    return s, t, N


def pairwise_counts_fast(rankings: np.ndarray) -> tuple[np.ndarray, np.ndarray, int]:
    """Vectorised version of pairwise_counts."""
    N, n = rankings.shape
    # r[:, i, None] < r[:, None, j] => item i ranked above j
    above = rankings[:, :, None] < rankings[:, None, :]  # (N, n, n)
    s = above.sum(axis=0)  # (n, n)
    t = np.zeros((n, n), dtype=int)
    return s, t, N


# 2b. Bradley-Terry MLE for alpha -----------------------------------------

def bradley_terry_mle(
    s: np.ndarray,
    max_iter: int = 500,
    tol: float = 1e-8,
) -> np.ndarray:
    """
    Maximum-likelihood estimation of Bradley-Terry parameters lambda.

    Uses the iterative algorithm of Hunter (2004):
        lambda_i^{new} = W_i / sum_{j != i} (s_ij + s_ji) / (lambda_i + lambda_j)

    where W_i = sum_{j} s_ij.

    Parameters
    ----------
    s : ndarray (n, n)
        s[i,j] = number of times i is ranked above j.

    Returns
    -------
    lam : ndarray (n,)
        Estimated BT parameters (normalised to sum to n).
    """
    n = s.shape[0]
    lam = np.ones(n)

    for _ in range(max_iter):
        lam_old = lam.copy()
        for i in range(n):
            W_i = s[i].sum()
            denom = 0.0
            for j in range(n):
                if j == i:
                    continue
                n_ij = s[i, j] + s[j, i]
                if n_ij > 0:
                    denom += n_ij / (lam[i] + lam[j])
            if denom > 0:
                lam[i] = W_i / denom
        # normalise
        lam *= n / lam.sum()
        if np.max(np.abs(lam - lam_old)) < tol:
            break
    return lam


def bt_alpha(lam: np.ndarray) -> np.ndarray:
    """
    Convert BT parameters to pairwise probabilities.
    alpha[i,j] = P(i > j | T_ij=0) = lambda_i / (lambda_i + lambda_j).
    """
    n = len(lam)
    alpha = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j:
                alpha[i, j] = 0.5
            else:
                alpha[i, j] = lam[i] / (lam[i] + lam[j])
    return alpha


# 2c. Pitman-Yor prior for pi_1 -------------------------------------------

def sample_pitman_yor_partition(
    n: int,
    delta: float,
    lam: float,
    rng: np.random.Generator | None = None,
) -> list[int]:
    """
    Sample one partition (block sizes) from a Pitman-Yor process.

    Parameters
    ----------
    n : int
        Number of items.
    delta : float
        Discount parameter, 0 <= delta <= 1.
    lam : float
        Strength parameter, lam > -delta.

    Returns
    -------
    block_sizes : list[int]
        Sizes of the blocks in the partition.
    """
    if rng is None:
        rng = np.random.default_rng()

    blocks: list[int] = []
    for i in range(1, n + 1):
        K = len(blocks)
        total = lam + i - 1
        probs = []
        # probability of joining each existing block
        for k in range(K):
            probs.append(max(blocks[k] - delta, 0.0))
        # probability of starting a new block
        probs.append(lam + K * delta)
        probs = np.array(probs)
        probs /= probs.sum()
        choice = rng.choice(len(probs), p=probs)
        if choice < K:
            blocks[choice] += 1
        else:
            blocks.append(1)
    return blocks


def estimate_pi1_pitman_yor(
    n: int,
    delta: float,
    lam: float,
    n_samples: int = 5000,
    rng: np.random.Generator | None = None,
) -> float:
    """
    Estimate the prior tie probability pi_1 by Monte-Carlo sampling
    partitions from a Pitman-Yor process:

        pi_1 = E[ sum_k C(|B_k|, 2) / C(n, 2) ]

    (Eq. 22 in the notes.)

    Parameters
    ----------
    n : int
        Number of items.
    delta, lam : float
        Pitman-Yor parameters.
    n_samples : int
        Number of MC samples.

    Returns
    -------
    pi1 : float
        Estimated prior probability that a pair is tied.
    """
    if rng is None:
        rng = np.random.default_rng(42)
    total_pairs = comb(n, 2, exact=True)
    if total_pairs == 0:
        return 0.0
    pi1_sum = 0.0
    for _ in range(n_samples):
        blocks = sample_pitman_yor_partition(n, delta, lam, rng)
        tied_pairs = sum(comb(bk, 2, exact=True) for bk in blocks)
        pi1_sum += tied_pairs / total_pairs
    return pi1_sum / n_samples


# 2d. Posterior tie probability q_ij ---------------------------------------

def log_binom_pmf(k: int, N: int, p: float) -> float:
    """Log of Binomial(k; N, p) PMF, handling edge cases."""
    if p <= 0.0:
        return 0.0 if k == 0 else -np.inf
    if p >= 1.0:
        return 0.0 if k == N else -np.inf
    return gammaln(N + 1) - gammaln(k + 1) - gammaln(N - k + 1) + k * np.log(p) + (N - k) * np.log(1 - p)


def log_beta_binomial_pmf(k: int, N: int, a: float, b: float) -> float:
    """
    Log of the Beta-Binomial(k; N, a, b) PMF:
        C(N,k) * B(k+a, N-k+b) / B(a, b)
    where B is the Beta function.

    This is the marginal likelihood of observing k successes in N trials
    when the success probability is integrated out under a Beta(a, b) prior.
    """
    from scipy.special import betaln
    return (gammaln(N + 1) - gammaln(k + 1) - gammaln(N - k + 1)
            + betaln(k + a, N - k + b) - betaln(a, b))


def posterior_tie_probability(
    s_ij: int,
    N: int,
    alpha_ij: float,
    pi1: float,
    method: str = "marginal",
    a_alpha: float = 1.0,
    b_alpha: float = 1.0,
) -> float:
    """
    Compute q_ij = P(T_ij = 1 | s_ij, N).

    Two methods:
    - "point": Uses the BT point estimate for alpha (eq. 24 directly).
      This tends to underestimate tie probability because BT adapts
      alpha to the observed ratio, making the strict hypothesis
      always look good.
    - "marginal" (recommended): Integrates out alpha under a Beta(a, b)
      prior for the strict hypothesis. The tied hypothesis still uses
      Binom(N, 0.5). This gives:
        p(s_ij | T=1) = C(N, s_ij) * 2^{-N}
        p(s_ij | T=0) = BetaBinomial(s_ij; N, a_alpha, b_alpha)

    Parameters
    ----------
    s_ij : int
        Number of times i ranked above j.
    N : int
        Number of assessors.
    alpha_ij : float
        BT probability P(i > j | not tied). Used only if method="point".
    pi1 : float
        Prior probability the pair is tied.
    method : str
        "marginal" or "point".
    a_alpha, b_alpha : float
        Beta prior parameters on alpha for the strict hypothesis
        (only used if method="marginal").

    Returns
    -------
    float
        Posterior probability q_ij in [0, 1].
    """
    # Tied hypothesis: s_ij ~ Binom(N, 0.5)
    log_tied = np.log(pi1 + 1e-300) + log_binom_pmf(s_ij, N, 0.5)

    if method == "point":
        log_strict = np.log(1 - pi1 + 1e-300) + log_binom_pmf(s_ij, N, alpha_ij)
    elif method == "marginal":
        log_strict = (np.log(1 - pi1 + 1e-300)
                      + log_beta_binomial_pmf(s_ij, N, a_alpha, b_alpha))
    else:
        raise ValueError(f"Unknown method: {method}")

    log_denom = np.logaddexp(log_tied, log_strict)
    q = np.exp(log_tied - log_denom)
    return float(q)


def compute_all_qij(
    s: np.ndarray,
    N: int,
    alpha: np.ndarray,
    pi1: float,
    method: str = "marginal",
    a_alpha: float = 1.0,
    b_alpha: float = 1.0,
) -> np.ndarray:
    """Compute q_ij for all i < j pairs."""
    n = s.shape[0]
    q = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            q[i, j] = posterior_tie_probability(
                s[i, j], N, alpha[i, j], pi1,
                method=method, a_alpha=a_alpha, b_alpha=b_alpha,
            )
            q[j, i] = q[i, j]
    return q


# 2e. Minimise misclassification loss to find p-hat -----------------------

def empirical_tie_weight(s: np.ndarray, N: int) -> np.ndarray:
    """w_ij = min(r_ij, 1 - r_ij), with r_ij = s_ij / N."""
    n = s.shape[0]
    r = s / max(N, 1)
    w = np.minimum(r, 1.0 - r)
    return w


def misclassification_loss(
    p: float,
    q: np.ndarray,
    w: np.ndarray,
) -> float:
    """
    Smooth misclassification loss (eq. 26):
        L(p) = sum_{i<j} (1{p < w_ij} - q_ij)^2
    """
    n = q.shape[0]
    loss = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            indicator = 1.0 if p < w[i, j] else 0.0
            loss += (indicator - q[i, j]) ** 2
    return loss


def estimate_p_data_informed(
    rankings: np.ndarray,
    pi1: float | None = None,
    delta: float = 0.5,
    lam: float = 1.0,
    n_mc_pi1: int = 5000,
    loss: str = "smooth",
    grid_size: int = 1000,
    method: str = "marginal",
    a_alpha: float = 1.0,
    b_alpha: float = 1.0,
) -> dict:
    """
    Data-informed estimation of p for a single cluster.

    Parameters
    ----------
    rankings : ndarray (N, n)
        Strict rankings matrix.
    pi1 : float or None
        Prior tie probability. If None, estimated via Pitman-Yor MC.
    delta, lam : float
        Pitman-Yor parameters (used only if pi1 is None).
    n_mc_pi1 : int
        Number of MC samples for pi1 estimation.
    loss : str
        "smooth" for eq. 26, "hard" for eq. 25.
    grid_size : int
        Number of grid points to search over (0, 0.5).
    method : str
        "marginal" (recommended) or "point" for posterior tie computation.
        "marginal" integrates out alpha under Beta(a_alpha, b_alpha).
        "point" uses the BT MLE for alpha (eq. 24 directly).
    a_alpha, b_alpha : float
        Beta prior parameters on alpha for marginal method.

    Returns
    -------
    dict with keys:
        p_hat        : estimated p
        pi1          : prior tie probability used
        q            : (n, n) posterior tie probabilities
        alpha        : (n, n) BT pairwise probabilities
        bt_lambda    : (n,) BT parameters
        s            : (n, n) pairwise counts
        w            : (n, n) empirical tie weights
    """
    N_obs, n = rankings.shape

    # Pairwise counts
    s, _, N = pairwise_counts_fast(rankings)

    # BT parameters (used for point method and diagnostics)
    bt_lam = bradley_terry_mle(s)
    alpha = bt_alpha(bt_lam)

    # Prior tie probability
    if pi1 is None:
        pi1 = estimate_pi1_pitman_yor(n, delta, lam, n_mc_pi1)

    # Posterior tie probabilities
    q = compute_all_qij(s, N, alpha, pi1, method=method,
                        a_alpha=a_alpha, b_alpha=b_alpha)

    # Empirical tie weights
    w = empirical_tie_weight(s, N)

    # Grid search for p_hat
    p_grid = np.linspace(1e-6, 0.5 - 1e-6, grid_size)
    losses = np.array([misclassification_loss(p, q, w) for p in p_grid])
    best_idx = np.argmin(losses)
    p_hat = float(p_grid[best_idx])

    return {
        "p_hat": p_hat,
        "pi1": pi1,
        "q": q,
        "alpha": alpha,
        "bt_lambda": bt_lam,
        "s": s,
        "w": w,
        "loss_curve": (p_grid, losses),
    }


# ---------------------------------------------------------------------------
# 3. Multi-cluster estimation
# ---------------------------------------------------------------------------

def estimate_p_multicluster(
    rankings: np.ndarray,
    cluster_assignments: np.ndarray,
    delta: float = 0.5,
    lam: float = 1.0,
    n_mc_pi1: int = 5000,
    grid_size: int = 1000,
) -> dict:
    """
    Estimate p across multiple clusters. Each cluster gets its own
    p_hat, and the overall p is a weighted average (weighted by
    number of assessors in each cluster).

    Parameters
    ----------
    rankings : ndarray (N, n)
        Strict rankings matrix (all assessors).
    cluster_assignments : ndarray (N,)
        Cluster label for each assessor.
    delta, lam : float
        Pitman-Yor parameters for pi1 estimation.
    n_mc_pi1 : int
        MC samples for pi1 estimation (per cluster).
    grid_size : int
        Grid resolution for p optimisation.

    Returns
    -------
    dict with keys:
        p_hat_global      : weighted average p
        p_hat_per_cluster  : dict {cluster_label: p_hat}
        weights            : dict {cluster_label: weight}
        details            : dict {cluster_label: full result dict}
    """
    clusters = np.unique(cluster_assignments)
    N_total = len(cluster_assignments)
    n_items = rankings.shape[1]

    cluster_results = {}
    p_hats = {}
    weights = {}

    for c in clusters:
        mask = cluster_assignments == c
        R_c = rankings[mask]
        N_c = R_c.shape[0]

        if N_c < 2:
            warnings.warn(f"Cluster {c} has < 2 assessors; skipping.")
            continue

        # Each cluster may have a different tie structure, so we estimate
        # pi1 separately per cluster using Pitman-Yor
        result = estimate_p_data_informed(
            R_c,
            pi1=None,
            delta=delta,
            lam=lam,
            n_mc_pi1=n_mc_pi1,
            grid_size=grid_size,
        )
        cluster_results[c] = result
        p_hats[c] = result["p_hat"]
        weights[c] = N_c

    # Weighted average
    total_weight = sum(weights.values())
    if total_weight == 0:
        p_hat_global = 0.25  # fallback
    else:
        p_hat_global = sum(
            p_hats[c] * weights[c] for c in p_hats
        ) / total_weight

    return {
        "p_hat_global": p_hat_global,
        "p_hat_per_cluster": p_hats,
        "weights": {c: w / total_weight for c, w in weights.items()},
        "details": cluster_results,
    }


# ---------------------------------------------------------------------------
# 4. Simple CI estimate for multi-cluster (no data needed)
# ---------------------------------------------------------------------------

def estimate_p_simple_multicluster_weighted(
    cluster_sizes: list[int],
) -> dict:
    """
    Simple CI estimate per cluster, weighted-averaged.

    Parameters
    ----------
    cluster_sizes : list[int]
        Number of assessors in each cluster.

    Returns
    -------
    dict with p_hat_global and per-cluster values.
    """
    total = sum(cluster_sizes)
    p_hats = {}
    for i, N_c in enumerate(cluster_sizes):
        p_hats[i] = estimate_p_simple(max(N_c, 2))
    p_global = sum(p_hats[i] * cluster_sizes[i] for i in range(len(cluster_sizes))) / total
    return {
        "p_hat_global": p_global,
        "p_hat_per_cluster": p_hats,
    }


# ---------------------------------------------------------------------------
# 5. Demo / example usage
# ---------------------------------------------------------------------------

def _generate_rankings_from_weak_order(
    weak_order: list[list[int]],
    N: int,
    noise: float = 0.1,
    rng: np.random.Generator | None = None,
) -> np.ndarray:
    """
    Generate N strict rankings from a weak-order consensus.

    Items in the same block are randomly shuffled; with probability
    `noise`, adjacent items across blocks are swapped.

    Parameters
    ----------
    weak_order : list[list[int]]
        Blocks ordered best-to-worst, e.g. [[0,1], [2], [3,4,5]]
        means items 0,1 are tied at rank 1, item 2 at rank 2, etc.
    N : int
        Number of rankings to generate.
    noise : float
        Probability of swapping adjacent items across blocks.

    Returns
    -------
    rankings : ndarray (N, n)
    """
    if rng is None:
        rng = np.random.default_rng(42)

    n = sum(len(b) for b in weak_order)
    rankings = np.zeros((N, n), dtype=int)

    for l in range(N):
        # Build a strict ordering by shuffling within blocks
        order = []
        for block in weak_order:
            shuffled = list(block)
            rng.shuffle(shuffled)
            order.extend(shuffled)

        # Apply noise: swap adjacent items with some probability
        for idx in range(len(order) - 1):
            if rng.random() < noise:
                order[idx], order[idx + 1] = order[idx + 1], order[idx]

        # Convert item ordering to ranks
        rank = np.zeros(n, dtype=int)
        for pos, item in enumerate(order):
            rank[item] = pos + 1
        rankings[l] = rank

    return rankings


    demo()

  Estimation of p for K(p) distance — Weak Mallows Model

--- 1. Simple confidence-interval estimate ---
  Formula: p = (sqrt(N) - 1.96) / (2*sqrt(N))
  Cluster 0 (N=50): p = 0.3614
  Cluster 1 (N=30): p = 0.3211
  Weighted global:   p = 0.3463

--- 2. Data-informed estimate (Cluster 0, marginal method) ---
  pi_1 (Pitman-Yor prior):  0.2465
  p_hat:                     0.4004
  Posterior tie probabilities q_ij (truly tied pairs marked *):
    q(0,1) = 0.4112  (s_ij= 30, s_ji= 20, BT_alpha=0.601) *
    q(0,2) = 0.0000  (s_ij= 46, s_ji=  4, BT_alpha=0.923)
    q(3,4) = 0.5681  (s_ij= 22, s_ji= 28, BT_alpha=0.433) *
    q(0,5) = 0.0000  (s_ij= 50, s_ji=  0, BT_alpha=1.000)
    q(2,5) = 0.0000  (s_ij= 50, s_ji=  0, BT_alpha=0.998)

--- 2b. Data-informed estimate (Cluster 0, point BT method) ---
  p_hat (point):    0.4404
  Posterior tie probabilities (point method):
    q(0,1) = 0.106804 *
    q(0,2) = 0.000000
    q(3,4) = 0.186577 *
    q(0,5) = 0.000000
    q(2,5) = 0.000000
  Note: po

# Demo


In [44]:
def get_tied_pairs(weak_order):
    """Extract all truly tied pairs from a weak order (list of blocks).
    
    A weak order like [[0, 1], [2], [3, 4]] means items 0,1 are tied
    and items 3,4 are tied.
    """
    tied = set()
    for block in weak_order:
        for i in range(len(block)):
            for j in range(i + 1, len(block)):
                tied.add((min(block[i], block[j]), max(block[i], block[j])))
    return tied


def demo(weak_orders, rankings, clusters):
    """
    Run a demonstration with arbitrary clusters.

    Parameters
    ----------
    weak_orders : list of list-of-lists
        One weak order per cluster, e.g. [[[0,1],[2],[3,4]], [[0],[1],[2,3,4]]]
    rankings : np.ndarray
        Full ranking matrix (n_judges x n_items), all clusters stacked.
    clusters : np.ndarray
        1D array of cluster labels, one per row of rankings.
    """
    cluster_labels = np.unique(clusters)
    n_clusters = len(cluster_labels)
    n_items = rankings.shape[1]

    # Split into per-cluster sub-matrices
    ranking_matrices = {
        label: rankings[clusters == label, :]
        for label in cluster_labels
    }
    cluster_sizes = {label: R.shape[0] for label, R in ranking_matrices.items()}

    # Precompute tied pairs per cluster
    tied_pairs = {
        label: get_tied_pairs(wo)
        for label, wo in zip(cluster_labels, weak_orders)
    }

    print("=" * 65)
    print("  Estimation of p for K(p) distance — Weak Mallows Model")
    print("=" * 65)

    # 1. Simple CI estimate
    print("\n--- 1. Simple confidence-interval estimate ---")
    print("  Formula: p = (sqrt(N) - 1.96) / (2*sqrt(N))")
    for label in cluster_labels:
        size = cluster_sizes[label]
        p_val = estimate_p_simple(size)
        print(f"  Cluster {label} (N={size}): p = {p_val:.4f}")
    p_global = estimate_p_simple_multicluster_weighted(list(cluster_sizes.values()))
    print(f"  Weighted global:   p = {p_global['p_hat_global']:.4f}")

    # 2. Per-cluster data-informed estimates
    for label, wo in zip(cluster_labels, weak_orders):
        R = ranking_matrices[label]
        truly_tied = tied_pairs[label]
        print(f"\n--- 2. Data-informed estimate (Cluster {label}, marginal method) ---")
        print(f"  True weak order: {wo}")
        print(f"  Truly tied pairs: {sorted(truly_tied)}")

        result = estimate_p_data_informed(
            R, delta=0.5, lam=1.0, n_mc_pi1=5000, method="marginal"
        )
        print(f"  pi_1 (Pitman-Yor prior):  {result['pi1']:.4f}")
        print(f"  p_hat:                     {result['p_hat']:.4f}")

        print("  Posterior tie probabilities q_ij (truly tied pairs marked *):")
        for i in range(n_items):
            for j in range(i + 1, n_items):
                q_val = result['q'][i, j]
                if q_val > 0.1 or (i, j) in truly_tied:
                    marker = " *" if (i, j) in truly_tied else ""
                    print(f"    q({i},{j}) = {q_val:.4f}  "
                          f"(s_ij={result['s'][i,j]:3d}, s_ji={result['s'][j,i]:3d}, "
                          f"BT_alpha={result['alpha'][i,j]:.3f}){marker}")

    # 3. Multi-cluster data-informed estimate
    print("\n--- 3. Multi-cluster data-informed estimate ---")
    mc_result = estimate_p_multicluster(
        rankings, clusters,
        delta=0.5, lam=1.0, n_mc_pi1=5000, grid_size=1000,
    )
    print(f"  Cluster-level p_hat values:")
    for c, p_c in mc_result["p_hat_per_cluster"].items():
        w = mc_result["weights"][c]
        print(f"    Cluster {c}: p_hat = {p_c:.4f}  (weight = {w:.3f})")
    print(f"  Global weighted p_hat: {mc_result['p_hat_global']:.4f}")

    # Tie detection summary per cluster
    det = mc_result["details"]
    for label in cluster_labels:
        if label in det:
            truly_tied_c = tied_pairs[label]
            print(f"\n  Cluster {label} tie detection (true ties: {sorted(truly_tied_c)}):")
            for i in range(n_items):
                for j in range(i + 1, n_items):
                    q_val = det[label]["q"][i, j]
                    if q_val > 0.1 or (i, j) in truly_tied_c:
                        marker = " *" if (i, j) in truly_tied_c else ""
                        print(f"    q({i},{j}) = {q_val:.4f}{marker}")

    print("\n" + "=" * 65)
    print("  Done.")
    print("=" * 65)

In [86]:
C_true = 3
C = 6
n_items = 10
n_blocks_range=(3, 6)
n_assessors = 500

alpha = 1

true_blocks, true_tau, z_true, rankings = generate_mixture_data(
    n_assessors=n_assessors,
    seed=123,
    alpha=alpha,
    C = C_true,
    n_items=n_items,
    n_blocks_range=n_blocks_range
    )

In [87]:
ranking_matrices_raw = np.array(rankings)  # Shape: (n_assessors, n_items)
ranking_matrices = np.zeros_like(ranking_matrices_raw)
for assessor_idx in range(ranking_matrices_raw.shape[0]):
    perm = ranking_matrices_raw[assessor_idx]
    # perm[pos] = item, so we need: rank[item] = pos
    ranking_matrices[assessor_idx] = np.argsort(perm)

weak_orders = true_blocks
cluster_labels = np.array(z_true)

mc_result = estimate_p_multicluster(
        ranking_matrices, cluster_labels,
        delta=0.5, lam=1.0, n_mc_pi1=5000, grid_size=1000,
    )

In [88]:
det = mc_result["details"]
for c in range(C_true):
    if c in det:
        print(f"\n  Cluster {c} tie detection (true ties: {sorted(get_tied_pairs(true_blocks[c]))}):")
        for i in range(n_items):
            for j in range(i + 1, n_items):
                q_val = det[c]["q"][i, j]
                marker = " *" if (i, j) in get_tied_pairs(true_blocks[c]) else ""
                print(f"    q({i},{j}) = {q_val:.4f}{marker}")


  Cluster 0 tie detection (true ties: [(0, 2), (0, 3), (0, 5), (0, 8), (0, 9), (1, 4), (1, 6), (2, 3), (2, 5), (2, 8), (2, 9), (3, 5), (3, 8), (3, 9), (4, 6), (5, 8), (5, 9), (8, 9)]):
    q(0,1) = 0.0000
    q(0,2) = 0.7581 *
    q(0,3) = 0.7648 *
    q(0,4) = 0.0000
    q(0,5) = 0.5999 *
    q(0,6) = 0.0000
    q(0,7) = 0.0000
    q(0,8) = 0.7301 *
    q(0,9) = 0.6433 *
    q(1,2) = 0.0000
    q(1,3) = 0.0000
    q(1,4) = 0.5999 *
    q(1,5) = 0.0000
    q(1,6) = 0.7648 *
    q(1,7) = 0.0000
    q(1,8) = 0.0000
    q(1,9) = 0.0000
    q(2,3) = 0.7648 *
    q(2,4) = 0.0000
    q(2,5) = 0.7581 *
    q(2,6) = 0.0000
    q(2,7) = 0.0000
    q(2,8) = 0.7670 *
    q(2,9) = 0.7648 *
    q(3,4) = 0.0000
    q(3,5) = 0.7301 *
    q(3,6) = 0.0000
    q(3,7) = 0.0000
    q(3,8) = 0.7648 *
    q(3,9) = 0.6791 *
    q(4,5) = 0.0000
    q(4,6) = 0.7301 *
    q(4,7) = 0.0000
    q(4,8) = 0.0000
    q(4,9) = 0.0000
    q(5,6) = 0.0000
    q(5,7) = 0.0000
    q(5,8) = 0.7581 *
    q(5,9) = 0.6433 *


In [79]:

# Convert rankings from (position -> item) to (item -> rank/position)
# generate_mixture_data returns permutations where ranking[i] = item at position i
# But pairwise_counts_fast expects ranking[i] = position of item i

ranking_matrices_raw = np.array(rankings)  # Shape: (n_assessors, n_items)
# Each row is a permutation: position -> item

# Invert: create (item -> position) format
ranking_matrices = np.zeros_like(ranking_matrices_raw)
for assessor_idx in range(ranking_matrices_raw.shape[0]):
    perm = ranking_matrices_raw[assessor_idx]
    # perm[pos] = item, so we need: rank[item] = pos
    ranking_matrices[assessor_idx] = np.argsort(perm)

weak_orders = true_blocks
cluster_labels = np.array(z_true)

print("Data format check:")
print(f"  Raw ranking (assessor 0): {ranking_matrices_raw[0]}")
print(f"  Inverted ranking (assessor 0): {ranking_matrices[0]}")
print(f"  Should be an inverse permutation")
print()

demo(weak_orders, ranking_matrices, cluster_labels)


Data format check:
  Raw ranking (assessor 0): [ 8  0 12  7 14 24 10 22 15  6  5 16  1 13 18 19  4 17 21  2 20 11  3 23  9]
  Inverted ranking (assessor 0): [ 1 12 19 22 16 10  9  3  0 24  6 21  2 13  4  8 11 17 14 15 20 18  7 23  5]
  Should be an inverse permutation

  Estimation of p for K(p) distance — Weak Mallows Model

--- 1. Simple confidence-interval estimate ---
  Formula: p = (sqrt(N) - 1.96) / (2*sqrt(N))
  Cluster 0 (N=1641): p = 0.4758
  Cluster 1 (N=1741): p = 0.4765
  Cluster 2 (N=1618): p = 0.4756
  Weighted global:   p = 0.4760

--- 2. Data-informed estimate (Cluster 0, marginal method) ---
  True weak order: [[13, 9, 0, 17, 14], [18, 7], [6, 21, 16, 3], [2, 20, 15], [10, 19, 12], [22, 24], [8, 11], [4], [23, 5], [1]]
  Truly tied pairs: [(0, 9), (0, 13), (0, 14), (0, 17), (2, 15), (2, 20), (3, 6), (3, 16), (3, 21), (5, 23), (6, 16), (6, 21), (7, 18), (8, 11), (9, 13), (9, 14), (9, 17), (10, 12), (10, 19), (12, 19), (13, 14), (13, 17), (14, 17), (15, 20), (16, 21), (2

In [69]:

# Check separation between tied and non-tied pairs
print(f"\n=== Separation Analysis ===\n")

tied_true = get_tied_pairs(true_blocks[c])
non_tied_pairs = []
tied_pairs_list = []

for i in range(n_items):
    for j in range(i+1, n_items):
        if (i, j) in tied_true:
            tied_pairs_list.append(result['q'][i, j])
        else:
            non_tied_pairs.append(result['q'][i, j])

tied_array = np.array(tied_pairs_list)
non_tied_array = np.array(non_tied_pairs)

print(f"Tied pairs (N={len(tied_pairs_list)}):")
print(f"  Mean q_ij: {tied_array.mean():.4f}")
print(f"  Std:  {tied_array.std():.4f}")
print(f"  Min:  {tied_array.min():.4f}")
print(f"  Max:  {tied_array.max():.4f}")

print(f"\nNon-tied pairs (N={len(non_tied_pairs)}):")
print(f"  Mean q_ij: {non_tied_array.mean():.4f}")
print(f"  Std:  {non_tied_array.std():.4f}")
print(f"  Min:  {non_tied_array.min():.4f}")
print(f"  Max:  {non_tied_array.max():.4f}")

print(f"\nDiscrimination:")
print(f"  Difference in means: {tied_array.mean() - non_tied_array.mean():.4f}")

# Count classification accuracy if we threshold at 0.5
threshold = 0.5
tied_correct = (tied_array > threshold).sum()
non_tied_correct = (non_tied_array < threshold).sum()
total_correct = tied_correct + non_tied_correct
total = len(tied_pairs_list) + len(non_tied_pairs)

print(f"\nClassification accuracy (threshold=0.5):")
print(f"  Tied pairs correctly classified: {tied_correct}/{len(tied_pairs_list)} ({100*tied_correct/len(tied_pairs_list):.1f}%)")
print(f"  Non-tied pairs correctly classified: {non_tied_correct}/{len(non_tied_pairs)} ({100*non_tied_correct/len(non_tied_pairs):.1f}%)")
print(f"  Overall accuracy: {total_correct}/{total} ({100*total_correct/total:.1f}%)")



=== Separation Analysis ===

Tied pairs (N=22):
  Mean q_ij: 0.9050
  Std:  0.0458
  Min:  0.7881
  Max:  0.9382

Non-tied pairs (N=83):
  Mean q_ij: 0.0000
  Std:  0.0000
  Min:  0.0000
  Max:  0.0000

Discrimination:
  Difference in means: 0.9050

Classification accuracy (threshold=0.5):
  Tied pairs correctly classified: 22/22 (100.0%)
  Non-tied pairs correctly classified: 83/83 (100.0%)
  Overall accuracy: 105/105 (100.0%)


In [66]:
len(cluster_labels[cluster_labels==0])/len(cluster_labels), len(cluster_labels[cluster_labels==1])/len(cluster_labels), len(cluster_labels[cluster_labels==2])/len(cluster_labels)

(0.3343, 0.3463, 0.3194)